# 第 20 章　決定木

材料の配合と試験までの日数を指定し，その条件でのコンクリートの圧縮強度を予測します。これまでは，列の値を使って一つの式を作りました。この章では，学習の記録から区切りを選び，条件ごとに別の値を返す方法を試します。

前半は本文に合わせて上から実行します。後半の約 20 分では，記録を一行選び，木の条件を自分でたどります。同じ一行で異なる深さの予測値と誤差を確かめてから，検証全体の成績を読んでください。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV が手元にない場合は公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

## 本文と一緒に実行

### 配合と材齢

セメントはコンクリートの材料の一つです。水と反応して固まり，砂や砂利を結び付けます。表の材料の量はコンクリート 1 m³ 当たりの kg，材齢は作ってから試験するまでの日数，圧縮強度は MPa です。予測に渡すのは，その条件が指定された時点で分かる七つの材料の量と材齢です。 [セメント協会の解説](https://jcassoc.or.jp/tips03.html)

下の表では配合番号 3 の記録を選びました。この四行で，7 日と 28 日の強度の差と，270 日と 365 日の強度の差は，材齢が変わる幅に応じて大きくなっていますか。

In [ ]:
from pathlib import Path

import pandas as pd

file_name = "concrete-strength.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

concrete = pd.read_csv(csv_path)
record_columns = ["record_id", "recipe_id", "cement", "water", "age", "strength"]
example_recipe = concrete.loc[concrete["record_id"].isin([68, 60, 3, 4]), record_columns]
example_recipe.sort_values("age").round({"strength": 2})

### 配合を分けて比べる

まだ学習に使っていない配合を評価するため，材料七列が同じ行を配合ごとにまとめて分けました。7 日後の行だけで学習し，同じ配合の 28 日後の行で採点すると，どんな問いになるでしょうか。

最初の表で，学習・検証・保留の行数と配合数を確認します。保留の強度は候補比較に使いません。次のセルで入力の列を選び，学習平均を使う基準と線形回帰を，同じ検証の 208 行で採点します。誤差が前章の「件」から何に変わったかも確認してください。

In [ ]:
split_summary = concrete.groupby("split").agg(
    rows=("recipe_id", "size"),
    recipes=("recipe_id", "nunique"),
)
split_summary.loc[["train", "validation", "holdout"]]

In [ ]:
from sklearn.linear_model import LinearRegression

features = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate", "age",
]
train = concrete.loc[concrete["split"] == "train"]
valid = concrete.loc[concrete["split"] == "validation"]

X_train = train[features]
X_valid = valid[features]
y_train = train["strength"]
y_valid = valid["strength"]

In [ ]:
mean_strength = y_train.mean()
linear_model = LinearRegression().fit(X_train, y_train)
linear_prediction = linear_model.predict(X_valid)

baseline_mae = pd.Series({
    "Training mean": (y_valid - mean_strength).abs().mean(),
    "Linear regression": (y_valid - linear_prediction).abs().mean(),
})
baseline_mae.round(2)

### 六行で境界を選ぶ

次の六行は，材齢だけで予測する仕組みを見る架空の例です。

| 記録 | A | B | C | D | E | F |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 材齢（日） | 3 | 7 | 14 | 28 | 56 | 90 |
| 強度（MPa） | 5 | 10 | 15 | 30 | 35 | 40 |

分けずに六行の平均を使えば，一律 22.5 MPa です。材齢 21 日以下を左とすると，左は A・B・C の平均 10 MPa，右は D・E・F の平均 35 MPa になります。材齢がちょうど 21 日なら左です。この規則で，どこから始め，どこで予測が決まるかを本文の図で追ってください。

10.5 日以下で分ける場合は，左右にどの行が入りますか。右に進む C の予測と残差の二乗を求め，21 日を境界にした場合と比べます。次のセルは，各候補について左右の平均を出し，六行の残差の二乗を足します。どの境界の二乗和が一番小さいでしょうか。

In [ ]:
toy = pd.DataFrame({
    "age": [3, 7, 14, 28, 56, 90],
    "strength": [5, 10, 15, 30, 35, 40],
})
toy_scores = {}
for threshold in [5, 10.5, 21, 42, 73]:
    goes_left = toy["age"] <= threshold
    fitted_mean = toy.groupby(goes_left)["strength"].transform("mean")
    toy_scores[threshold] = ((toy["strength"] - fitted_mean) ** 2).sum()
pd.Series(toy_scores, name="sum_squared_errors").rename_axis("age_threshold")

### 深さ 2 の木

今度は，線形回帰で使った八つの入力をそのまま木に渡します。それぞれの分岐でどの列のどの境界を使うか，学習した木の図を読みます。`max_depth=2` の指定では，根から最大二回の分岐を通ると葉に着きます。

図の `samples` はその箱に来た学習行数，`value` は強度の平均です。最初に左へ行く場合と右へ行く場合で，次に出てくる材齢の境界を比べてください。

In [ ]:
from sklearn.tree import DecisionTreeRegressor

shallow_tree = DecisionTreeRegressor(
    criterion="squared_error", max_depth=2, random_state=2026,
)
shallow_tree.fit(X_train, y_train)
shallow_prediction = shallow_tree.predict(X_valid)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(8.5, 4.65), layout="constrained")
plot_tree(shallow_tree, feature_names=features, impurity=False,
          rounded=True, precision=2, fontsize=12, ax=ax)
plt.show()

### 三行をたどる

検証の 56 番と 12 番は同じ配合で，`cement = 198.6` です。材齢はそれぞれ 7 日，28 日です。1 番はセメント量 540 kg/m³ で，材齢 28 日です。三行がどの葉に進むか，図の上から順にたどります。次の表の予測列 `shallow` と照合してください。

領域の図では，横の位置がセメント量，縦の位置が材齢です。線で分かれた領域と，先ほどたどった箱の `value` がどう対応するかを読みます。

In [ ]:
scored = valid.assign(shallow=shallow_prediction)
shown_records = [56, 12, 1]
example_columns = ["recipe_id", "cement", "age", "strength", "shallow"]
examples = scored.set_index("record_id").loc[shown_records, example_columns]
examples.round(2)

In [ ]:
from matplotlib.patches import Rectangle

cement_cut, age_cut_low, age_cut_high = shallow_tree.tree_.threshold[[0, 1, 4]]
region_bounds = [
    (100, cement_cut, 0, age_cut_low, 2, "#dcebf2"),
    (100, cement_cut, age_cut_low, 70, 3, "#bad8e8"),
    (cement_cut, 560, 0, age_cut_high, 5, "#aed0e0"),
    (cement_cut, 560, age_cut_high, 70, 6, "#79b5d1"),
]
fig, ax = plt.subplots(figsize=(7.2, 4.2), layout="constrained")
for x0, x1, y0, y1, leaf, color in region_bounds:
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0,
                           facecolor=color, edgecolor="#385365"))
    value = shallow_tree.tree_.value[leaf, 0, 0]
    ax.text(x0 + 0.67 * (x1 - x0), (y0 + y1) / 2, f"{value:.2f} MPa",
            ha="center", va="center", color="#172e40")

ax.scatter(examples["cement"], examples["age"], color="#a7421f",
           edgecolors="white", s=46, zorder=3)
for record_id, row in examples.iterrows():
    ax.annotate(f"#{record_id}", (row["cement"], row["age"]),
                xytext=(0, -8), textcoords="offset points", ha="center", va="top")
ax.set(xlabel="Cement (kg/m³)", ylabel="Age (days)",
       xlim=(100, 560), ylim=(0, 70), xticks=[100, 200, 300, 352.5, 450, 540],
       yticks=[0, 10.5, 21, 28, 56, 70])
plt.show()

### 深さと誤差

深さの上限を増やして同じ学習入力で木を作ります。出力の表では，指定した上限と，実際の深さを見比べてください。図では学習の MAE と検証の MAE を別々に読みます。深くするにつれて学習での外れはどのくらい減り，検証での外れはどう変わりましたか。線形回帰の検証 MAE は点線で表します。

In [ ]:
trees = {2: shallow_tree}
for depth in [3, 5, None]:
    trees[depth] = DecisionTreeRegressor(
        criterion="squared_error", max_depth=depth, random_state=2026,
    ).fit(X_train, y_train)

rows = []
for depth, model in trees.items():
    train_error = (y_train - model.predict(X_train)).abs()
    valid_error = (y_valid - model.predict(X_valid)).abs()
    rows.append({
        "depth_limit": str(depth),
        "actual_depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "train_mae": train_error.mean(),
        "valid_mae": valid_error.mean(),
    })
depth_scores = pd.DataFrame(rows).set_index("depth_limit")
depth_scores.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.7), layout="constrained")
ax.plot(depth_scores.index, depth_scores["train_mae"], color="#195b8a",
        marker="o", label="Tree: training")
ax.plot(depth_scores.index, depth_scores["valid_mae"], color="#a7421f",
        marker="s", linestyle="--", label="Tree: validation")
ax.axhline(baseline_mae["Linear regression"], color="#5a6670",
           linestyle=":", label="Linear regression: validation")
ax.set(xlabel="Maximum depth (None = unlimited)", ylabel="MAE (MPa)",
       ylim=(-0.32, 11), yticks=[0, 2, 4, 6, 8, 10])
ax.legend(frameon=False, loc="lower left")
plt.show()

### 一行と全体

56 番，12 番，1 番に戻り，浅い木と深さを制限しない木の予測を並べます。56 番と 12 番は同じ配合でも材齢が違いました。各行の実測と比べて，どちらの予測が近いかを確かめます。その一行の結果と 208 行の MAE が何を示すか，それぞれ説明できるようにしましょう。

In [ ]:
scored["unlimited"] = trees[None].predict(X_valid)
scored["abs_shallow"] = (scored["strength"] - scored["shallow"]).abs()
scored["abs_unlimited"] = (scored["strength"] - scored["unlimited"]).abs()

comparison_columns = ["strength", "shallow", "unlimited", "abs_shallow", "abs_unlimited"]
scored.set_index("record_id").loc[[56, 12, 1], comparison_columns].round(2)

## 実習（約 20 分）

### 1. 一行を選んで分岐をたどる（5 分）

次の四行から一行選びます。どれも本文でたどった三行とは別の検証行です。実測はまだ表示しません。セメント量と材齢を使って，浅い木で最初にどちらへ進むか，次はどちらの条件を答えるかをたどり，葉の予測値を読みます。

**選んだ行番号，最初と次の分岐，葉の予測値**　…

In [ ]:
practice_records = [82, 131, 480, 402]
valid.set_index("record_id").loc[practice_records, ["cement", "age"]]

### 2. 同じ行で二つの木を比べる（8 分）

次のセルで `record_choice` を選んだ行番号に替えます。まずは `depth_limit = None` のままで，浅い木と，深さを制限しない木を比べます。浅い木の予測が自分でたどった値と一致するかを確かめてから，二つの予測と実測の差を読みます。続けて，どちらの検証全体の MAE が小さいかを確かめます。

余裕があれば，先にその行でどちらの木に近い予測が出そうかを考え，`depth_limit` を 3 か 5 に一度だけ替えてみます。

In [ ]:
record_choice = None  # 選んだ行番号に置き換える
depth_limit = None  # None は深さを制限しない

if record_choice is None:
    print("選んだ行番号を record_choice に指定してください。")
else:
    chosen = valid.loc[valid["record_id"] == record_choice]
    chosen_X = chosen[features]
    comparison = chosen[["record_id", "strength"]].copy()
    comparison["shallow"] = shallow_tree.predict(chosen_X)
    comparison["deeper"] = trees[depth_limit].predict(chosen_X)
    comparison["abs_shallow"] = (comparison["strength"] - comparison["shallow"]).abs()
    comparison["abs_deeper"] = (comparison["strength"] - comparison["deeper"]).abs()
    display(comparison.round(2))
    display(depth_scores.loc[["2", str(depth_limit)], ["valid_mae"]].round(2))

### 3. 採用する条件を考える（7 分）

選んだ一行で二つの木が返した予測値と絶対誤差を挙げ，どちらが近かったかを述べます。208 行全体の MAE でも同じ順だったかを確かめます。その上で，新しい配合の強度を見積もるときにどちらを候補にするかと，追加で確かめたい配合や材齢を一つ，3〜4 文で書いてください。

**結果と次に確かめたいこと**　…

提出する場合は，報告に使った行番号と設定をセルに指定します。カーネルを再起動して上から全セルを実行し，分岐の説明，集計表，報告が残る状態でノートを保存してください。

### データの出典

I-Cheng Yeh が公開した *Concrete Compressive Strength* を使用します。配布用 CSV は列名を変更し，行番号，配合の番号，分割を追加しています。 [データ DOI](https://doi.org/10.24432/C5PK67) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)